In [38]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [39]:
X = pd.read_csv("train.csv")
y = X.pop("label")

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

In [40]:
class Layer:
    def __call__(self, prev):
        return self.forward(prev)
    
    def forward(self, prev):
        raise NotImplementedError

    def backward(self, dx):
        pass

    def grad_descent(self, alpha):
        if self.prev_layer:
            self.prev_layer.grad_descent(alpha)

In [41]:
class Flatten(Layer):
    def __init__(self):
        super().__init__()

    def forward(self, prev):
        self.prev_layer = prev if isinstance(prev, Layer) else None
        self.prev_x = prev.x if self.prev_layer is not None else prev

        self.prev_shape = self.prev_x.shape
        self.x = self.prev_x.reshape(self.prev_x.shape[0], -1)
        
        return self

    def backward(self, dx):
        if not self.prev_layer: return

        self.prev_layer.backward(dx.reshape(self.prev_shape))

In [42]:
class ReLU(Layer):
    def __init__(self):
        super().__init__()

    def forward(self, prev):
        self.x = np.maximum(prev.x, 0)
        self.prev_layer = prev
        
        return self
    
    def backward(self, dx):
        if not self.prev_layer: return
        
        self.prev_layer.backward(
            dx * (self.prev_layer.x > 0)
        )

In [43]:
class Linear(Layer):
    def __init__(self, n_in, n_out):
        super().__init__()

        self.w = np.random.randn(n_in, n_out) * np.sqrt(2 / n_in)
        self.b = np.zeros(n_out)

    def forward(self, prev):
        self.prev_layer = prev
        self.x = self.prev_layer.x @ self.w + self.b

        return self
    
    def backward(self, dx):
        self.db = np.sum(dx, axis=0) / dx.shape[0]

        if not self.prev_layer: return

        self.dw = self.prev_layer.x.T @ dx / dx.shape[0]
        self.prev_layer.backward(dx @ self.w.T)

    def grad_descent(self, alpha):
        self.w -= self.dw*alpha
        self.b -= self.db*alpha

        super().grad_descent(alpha)


In [ ]:
from scipy import signal

class Conv2D(Layer):
    def __init__(self, in_channels, out_channels, kernel_size=3, padding=1):
        super().__init__()

        fan_in = kernel_size*kernel_size * in_channels
        std = np.sqrt(2.0/fan_in)

        self.padding = padding

        self.kernel = np.random.randn(in_channels, out_channels, kernel_size, kernel_size) * std
        self.bias = np.zeros((out_channels, 1, 1))

    def forward(self, prev):
        self.prev_layer = prev if isinstance(prev, Layer) else None
        self.prev_x = prev.x if self.prev_layer is not None else prev
        batches, _, prev_x_h, prev_x_w = self.prev_x.shape

        in_channels, out_channels, _, _ = self.kernel.shape

        self.x = np.zeros((batches, out_channels, prev_x_h, prev_x_w))

        for b in range(batches):
            for out_c in range(out_channels):
                for in_c in range(in_channels):
                    prev_x_padded = np.pad(
                        self.prev_x[b, in_c], 
                        self.padding
                    )

                    self.x[b, out_c] += signal.correlate2d(
                        prev_x_padded,
                        self.kernel[in_c, out_c],
                        mode='valid',
                    )

        self.x += self.bias[None]

        return self

    def backward(self, dx):
        in_channels, out_channels, k_size, _ = self.kernel.shape

        batches = dx.shape[0]
        self.dkernel = np.zeros_like(self.kernel)
        self.dbias = np.sum(dx, axis=(0, 2, 3)).reshape(out_channels, 1, 1) / batches

        for b in range(batches):
            for out_c in range(out_channels):
                for in_c in range(in_channels):
                    prev_x_padded = np.pad(
                        self.prev_x[b, in_c], 
                        self.padding
                    )

                    self.dkernel[in_c, out_c] += signal.correlate2d(
                        prev_x_padded, dx[b, out_c], mode='valid'
                    )

        self.dkernel /= batches

        if not self.prev_layer: return

        prev_dx = np.zeros_like(self.prev_layer.x)
        for b in range(batches):
            for out_c in range(out_channels):
                for in_c in range(in_channels):
                    prev_dx[b, in_c] += signal.convolve2d(
                        dx[b, out_c], self.kernel[in_c, out_c], mode='same'
                    )

        self.prev_layer.backward(prev_dx)

    def grad_descent(self, alpha):
        self.kernel -= self.dkernel*alpha
        self.bias -= self.dbias*alpha

        super().grad_descent(alpha)
    

In [45]:
class MaxPool2D(Layer):
    def __init__(self, size=2):
        super().__init__()

        self.size=size

    def forward(self, prev):
        self.prev_layer = prev
        prev_x = prev.x

        size = self.size

        batches, channels, width, height = prev_x.shape
        new_width = -(width // -size)
        new_height = -(height // -size)

        self.x = np.zeros((
            batches,
            channels,
            new_width,
            new_height
        ))

        self.mask = np.zeros_like(prev_x)

        for b in range(batches):
            for c in range(channels):
                for i in range(new_width):
                    for j in range(new_height):
                        w_start = i*size
                        h_start = j*size

                        region = prev_x[b, c, w_start:w_start+size, h_start:h_start+size]
                        self.x[b, c, i, j] = np.max(region)

                        local_r, local_c = np.unravel_index(np.argmax(region), region.shape)

                        self.mask[b, c, w_start+local_r, h_start+local_c] = 1

        return self

    def backward(self, dx):
        size = self.size
        batches, channels, new_width, new_height = self.x.shape

        prev_dx = np.zeros_like(self.mask)

        for b in range(batches):
            for c in range(channels):
                for i in range(new_width):
                    for j in range(new_height):
                        w_start = i*size
                        h_start = j*size

                        prev_dx[b, c, w_start:w_start+size, h_start:h_start+size] = (
                            dx[b, c, i, j] * self.mask[b, c, w_start:w_start+size, h_start:h_start+size]
                        )
        self.prev_layer.backward(prev_dx)


In [46]:
def softmax(z):
    z = np.asarray(z)
    e_z = np.exp(z - np.max(z, axis=1, keepdims=True))
    return e_z / np.sum(e_z, axis=1, keepdims=True)

class CrossEntropy():
    def __init__(self, logits_linear, y):
        self.y = y
        self.logits_linear = logits_linear
        self.y_cap = softmax(logits_linear.x)

        self.value = -np.sum(self.y*np.log(self.y_cap + 1e-15))

    def backward(self):
        self.logits_linear.backward(self.y_cap - self.y)

    def grad_descent(self, alpha):
        self.logits_linear.grad_descent(alpha)

In [47]:
def one_hot(y, num_classes=10):
    res = np.zeros((len(y), num_classes))
    res[np.arange(len(y)), y] = 1
    return res

In [48]:
class CNN:
    def __init__(self):
        self.sequence = [
            Conv2D(1, 8),
            ReLU(),
            MaxPool2D(2),

            Conv2D(8, 16),
            ReLU(),
            MaxPool2D(2),

            Flatten(),
            Linear(16*7*7, 128),
            ReLU(),
            Linear(128, 10)
        ]

    def forward(self, x):
        for layer in self.sequence:
            x = layer(x)

        return x

    def predict(self, Xt):
        Xt = np.asarray(Xt, dtype=np.float64) / 255.0
        logits_linear = self.forward(Xt)
        return np.argmax(logits_linear.x, axis=1)

    def fit(self, Xt, yt, epochs=50, alpha=0.1, batch_size=32, verbose=0):
        Xt = np.asarray(Xt, dtype=np.float64)/255.0
        yt = one_hot(yt)
        
        n = Xt.shape[0]

        rng = np.random.default_rng(42)

        for epoch in range(epochs):
            indices = rng.permutation(n)
            Xt = Xt[indices]
            yt = yt[indices]

            l = 0

            for start in range(0, n, batch_size):
                end = min(start + batch_size, n)
                X_batch = Xt[start:end]
                y_batch = yt[start:end]

                X_batch = X_batch.reshape(-1, 1, 28, 28)

                logits_linear = self.forward(X_batch)
                loss = CrossEntropy(logits_linear, y_batch)
                loss.backward()
                loss.grad_descent(alpha)

                l += loss.value

            if verbose and ((epoch+1)%verbose == 1 or epoch+1==epochs):
                print(f"epoch {epoch+1}/{epochs}. Loss: {l/n}")

        print("done.")

In [49]:
cnn = CNN()

cnn.fit(X_train, y_train, alpha=0.1, epochs=30, batch_size=64, verbose=1)

KeyboardInterrupt: 

In [ ]:
def accuracy(y_cap, y):
    return np.mean(y_cap==y)

y_cap = cnn.predict(X_val)
print(accuracy(y_cap, y_val))

0.97


In [ ]:
X_test = pd.read_csv("test.csv")
y_test_cap = cnn.predict(X_test)

res = pd.DataFrame({
    "ImageId": range(1, len(y_test_cap)+1),
    "Label": y_test_cap
})

res.to_csv("submission.csv", index=False)